# 7. TE mCG

Part of the **[Fig. 2 chapter](fig2.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import os
import numpy as np
import pandas as pd
from glob import glob
from scipy.sparse import csr_matrix
from scipy.stats import zscore
from concurrent.futures import ProcessPoolExecutor, as_completed

import anndata
from ALLCools.mcds import MCDS, RegionDS
from ALLCools.clustering import *
from ALLCools.plot import *

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")
import h5py
from anndata._io.specs import read_elem
from sklearn.metrics import adjusted_rand_score as ARI


In [ ]:
indir = f'{ENTEX_ROOT}/'
outdir = f'{indir}analysis/TE_mCG/'


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c35', 'c36'], axis=0)
# L1_meta = L1_meta.drop(['c7'], axis=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()


In [ ]:
import cooler
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
adata_list = np.sort(glob(f'{outdir}rlt*h5ad'))

In [ ]:
adata = anndata.read_h5ad(f'{ENTEX_ROOT}/analysis/TE_mCG/goodcov.raw.LINE.h5ad')
adata

In [ ]:
feat = ['all', 'gene2k', 'nonpeak', 'nongene2k', 'nongene2kpeak']
ari = []
cat = []
for adata_path in adata_list:
    with h5py.File(adata_path) as f:
        tmp = read_elem(f['obs'])
    flag = 0
    for f in feat:
        if f'leiden_{f}' not in tmp.keys():
            flag = 1
    if flag:
        continue
    cat.append(adata_path.split('/')[-1].split('.')[1])
    ari.append([ARI(tmp['majortype'], tmp[f'leiden_{f}']) for f in feat])
    print(adata_path)
    

In [ ]:
ari = pd.DataFrame(ari, index=cat, columns=feat)
ari = ari.loc[ari.mean(axis=1).sort_values().index[::-1]]
ari

In [ ]:
data = ari.stack().reset_index()
data.columns = ['Repeat Category', 'Feature Selection', 'ARI']


In [ ]:
fig, ax = plt.subplots(figsize=(6,2), dpi=300)
sns.barplot(data, x='Repeat Category', y='ARI', hue='Feature Selection', 
            order=ari.index, ax=ax, palette='muted')
ax.set_xticks(np.arange(ari.shape[0]))
ax.set_xticklabels(ari.index, rotation=90)
# fig.tight_layout()
fig.savefig(f'{outdir}TE_ARI.pdf', transparent=True)


In [ ]:
ds = 0.2
coord_base = 'tsne'
fig, axes = plt.subplots(4, len(feat), figsize=(4*len(feat),3*4), dpi=300, constrained_layout=True)
for i,t in enumerate(['DNA', 'LINE', 'LTR', 'SINE']):
    adata_path = f'{outdir}rlt.{t}.h5ad'
    with h5py.File(adata_path) as f:
        obs = read_elem(f['obs'])
        obsm = read_elem(f['obsm'])
    for j,f in enumerate(feat):
        obs[['tsne_0', 'tsne_1']] = obsm[f'X_tsne_{f}'].copy()
        ax = axes[i,j]
        _ = categorical_scatter(data=obs,
                                ax=ax,
                                coord_base=coord_base,
                                hue='majortype',
                                s=ds,
                                labelsize=8,
                                max_points=None,
                                palette=L1_color,
                                scatter_kws={'rasterized':True},
                               )

for f,ax in zip(feat, axes[0]):
    ax.set_title(f+'\n')
    
for t,ax in zip(['DNA', 'LINE', 'LTR', 'SINE'], axes[:,2]):
    ax.set_title(t)

axes[0,2].set_title('nonpeak\nDNA')
fig.savefig(f'{outdir}TE_tsne.pdf', transparent=True)


In [ ]:
allcools generate-dataset --allc_table allclist_L1.tsv \
    --output_path L1_repeat.mcds \
    --chrom_size_path /large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes \
    --obs_dim L1 \
    --cpu 36 \
    --chunk_size 1 \ 
    --regions SINE /large_storage/zhoulab/ref/repeatmasker/hg38/hg38.repeatmasker.repClass-SINE.bed \
    --regions LINE /large_storage/zhoulab/ref/repeatmasker/hg38/hg38.repeatmasker.repClass-LINE.bed \
    --regions LTR /large_storage/zhoulab/ref/repeatmasker/hg38/hg38.repeatmasker.repClass-LTR.bed \
    --quantifiers SINE count CGN \
    --quantifiers LINE count CGN \
    --quantifiers LTR count CGN


In [ ]:
import pyranges as pr

gene_bed_path = f'{REF_ROOT}/hg38/gencode/v30/gencode.v30.annotation.gene.sorted.bed.gz'
peak_bed_path = f'{indir}scATAC/peak/majortype/merged_peak.bed'

In [ ]:
for var_dim in ['SINE', 'LINE', 'LTR']:
    mcds = MCDS.open(f'{outdir}L1_repeat.mcds', obs_dim='L1', var_dim=var_dim)
    # mcds.add_feature_cov_mean(var_dim='chrom1k', plot=False)
    cov = mcds[f'{var_dim}_da'].sel(count_type='cov').mean(dim='L1').squeeze().to_pandas()
    mcds = mcds.sel({var_dim:cov.index[cov>20]})
    black_list_path = '/large_experiments/zhoulab/ref/blacklist/hg38-blacklist.v2.bed.gz'
    mcds = mcds.remove_black_list_region(black_list_path=black_list_path, f=0.5)
    chromosome_to_remove = ['chrX', 'chrY', 'chrM', 'chrL']
    mcds = mcds.remove_chromosome(chromosome_to_remove)
    mcds.add_mc_frac(normalize_per_cell=True, clip_norm_value=10)
    data = mcds[f'{var_dim}_da_frac'].to_pandas()
    data.to_hdf(f'{outdir}L1_{var_dim}_mCG.hdf', key='data')


In [ ]:
data_all = {}
for var_dim in ['SINE', 'LINE', 'LTR']:
    data = pd.read_hdf(f'{outdir}L1_{var_dim}_mCG.hdf', key='data')
    mcds = MCDS.open(f'{outdir}L1_repeat.mcds', obs_dim='L1', var_dim=var_dim)
    bed = mcds[[f'{var_dim}_chrom', f'{var_dim}_start', f'{var_dim}_end']].to_pandas()
    bed = bed.loc[data.columns]
    bed.columns = ['Chromosome', 'Start', 'End', 'mc_type']
    bed = bed.drop('mc_type', axis=1).reset_index()
    gr = pr.PyRanges(bed)
    gr = gr.subtract(pr.read_bed(peak_bed_path))
    gr = gr.subtract(pr.read_bed(gene_bed_path).slack(2000))
    data = data.loc[:, gr.df[var_dim].values]
    if data.shape[1] > 5000:
        tmp = data.loc[:, data.std(axis=0).sort_values().index[-5000:]]
        # tmp = data.loc[:, np.random.choice(data.columns, 5000, False)]
    else:
        tmp = data.copy()
        
    cg = sns.clustermap(tmp, metric='cosine', cmap='Blues_r', 
                yticklabels=data.index.map(L1_annot), 
                z_score=1, vmin=-2, vmax=0, figsize=(0.1, 0.1),
                )
    rorder = cg.dendrogram_row.reordered_ind
    corder = cg.dendrogram_col.reordered_ind
    data_all[var_dim] = tmp.iloc[rorder, corder]



In [ ]:
vmin, vmax = -2, 0
fig, axes = plt.subplots(1, 3, figsize=(12,6), dpi=300)
for i,var_dim in enumerate(data_all.keys()):
    ax = axes[i]
    tmp = zscore(data_all[var_dim], axis=0)
    im = ax.imshow(tmp, aspect='auto', cmap='Blues_r', interpolation='none',
                   vmin=vmin, vmax=vmax, rasterized=True)
    # sns.heatmap(zscore(tmp, axis=0), 
    #             xticklabels=False, yticklabels=tmp.index.map(L1_annot), 
    #             cmap='Blues_r', ax=ax, vmin=vmin, vmax=vmax, rasterized=True,
    #             cbar_kws={"shrink": 0.2, "aspect": 5, "ticks": [vmin, vmax]})
    # sns.despine(ax=ax, top=False, right=False, left=False, bottom=False)
    ax.set_xticks([])
    ax.set_yticks(np.arange(tmp.shape[0]))
    ax.set_yticklabels(tmp.index.map(L1_annot), rotation=0)
    ax.set_title(var_dim)
    
# plt.colorbar(im, ax=ax, shrink=0.2, aspect=5, ticks=[vmin, vmax])
fig.tight_layout()
fig.savefig(f'{outdir}TE_nonpeak_nongene_heatmap.pdf', transparent=True)
